In [1]:
from ollama import chat
from pydantic import BaseModel

class Country(BaseModel):
  name: str
  capital: str
  languages: list[str]

response = chat(
  messages=[
    {
      'role': 'user',
      'content': 'Tell me about Canada.',
    }
  ],
  model='smollm2:latest',
  format=Country.model_json_schema(),
)

ResponseError: json: cannot unmarshal object into Go struct field ChatRequest.format of type string (status code: 400)

In [ ]:
country = Country.model_validate_json(response.message.content)
print(country)

In [4]:
!pip show ollama

In [1]:
from ollama import chat
from pydantic import BaseModel

class Pet(BaseModel):
  name: str
  animal: str
  age: int
  color: str | None
  favorite_toy: str | None

class PetList(BaseModel):
  pets: list[Pet]

In [9]:
PetList.model_json_schema()['$defs']

{'Pet': {'properties': {'name': {'title': 'Name', 'type': 'string'},
   'animal': {'title': 'Animal', 'type': 'string'},
   'age': {'title': 'Age', 'type': 'integer'},
   'color': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
    'title': 'Color'},
   'favorite_toy': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
    'title': 'Favorite Toy'}},
  'required': ['name', 'animal', 'age', 'color', 'favorite_toy'],
  'title': 'Pet',
  'type': 'object'}}

In [11]:
def _wrap_structured_output(query: str, 
                            structured_output: BaseModel) -> str:
  _query = f"""
    {query}
    
    Return only the following JSON:
    {structured_output.model_json_schema()}
    
    JSON:
    """
  return _query

user_query = """
        I have two pets.
        A cat named Luna who is 5 years old and loves playing with yarn. She has grey fur.
        I also have a 2 year old black cat named Loki who loves tennis balls."""
response = chat(
  messages=[
    {
      'role': 'user',
      'content': _wrap_structured_output(user_query, PetList)
    }
  ],
  model='llama3.2:3b',
  format=PetList.model_json_schema(),
)

print(response.message.content)

ResponseError: json: cannot unmarshal object into Go struct field ChatRequest.format of type string (status code: 400)

In [12]:
!pip show ollama
!ollama --version

ollama version is 0.3.14


In [67]:
pets = PetList.model_validate_json(response.message.content)
print(pets)


ValidationError: 1 validation error for PetList
pets
  Field required [type=missing, input_value={'$defs': {'Pet': {'prope...List', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [31]:
from ollama import chat
import json

# Simple chat example
response = chat(
    model="smollm2:latest",  # or any model you have pulled
    messages=[
        {
            'role': 'user',
            'content': 'Tell me a short joke',
        },
    ]
)

print(response['message']['content'])

Why don't scientists trust atoms? Because they make up everything! 😄


In [22]:
# Example with streaming
print("\nStreaming response:")
stream = chat(
    model="smollm2:latest",
    messages=[{'role': 'user', 'content': 'Tell me a short fact about space'}],
    stream=True
)

for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)


Streaming response:


Did you know that stars typically burn for around 4-5 billion years before they die? The Sun is roughly halfway through its life and will eventually exhaust its fuel.